### Healpix tutorial
Frollowing [this](https://gcn.nasa.gov/docs/sample/healpix) healpix tutorial.

In [1]:
import astropy_healpix as ah
import numpy as np
from astropy import units as u
from astropy.table import QTable
import os

In [2]:
map_dir = "/Users/cristozilleruelo/Downloads"
map_name_2 = "IceCube-251105E_skymap_probdensity_multiorder.fits.gz"
skymap = QTable.read(os.path.join(map_dir, map_name_2))
skymap.keys()

['UNIQ', 'PROBDENSITY']

Now the idea is to finds the point whith the maximum probability of being found. 

In [3]:
hp_index = np.argmax(skymap['PROBDENSITY'])
uniq = skymap[hp_index]['UNIQ']
uniq

np.int64(96803968)

In [4]:
level, ipix = ah.uniq_to_level_ipix(uniq)
nside = ah.level_to_nside(level)
ra, dec = ah.healpix_to_lonlat(ipix, nside, order='nested')
print("RA: ", ra, " / ", ra.to(u.deg))
print("DEC: ", dec, " / ", dec.to(u.deg))

RA:  2.559 rad  /  146d37m12.65434033s
DEC:  0.923459 rad  /  52d54m37.05436152s


We can get the probability density at a specific point as well:

In [7]:
ra, dec = ra.to(u.deg), dec.to(u.deg)

level, ipix = ah.uniq_to_level_ipix(skymap['UNIQ'])
nside = ah.level_to_nside(level)

match_ipix = ah.lonlat_to_healpix(ra, dec, nside, order='nested')

match_index = np.flatnonzero(ipix == match_ipix)[0]

prob_density = skymap[match_index]['PROBDENSITY'].to_value(u.deg**-2)
prob_density

AttributeError: 'numpy.float64' object has no attribute 'tovalue'

#### 90% probability region

In [6]:
#Sort the pixels by descending probability density
skymap.sort('PROBDENSITY', reverse=True)

#Area of each pixel
level, ipix = ah.uniq_to_level_ipix(skymap['UNIQ'])
pixel_area = ah.nside_to_pixel_area(ah.level_to_nside(level))

#Pixel area times the probability
prob = pixel_area * skymap['PROBDENSITY']

#Cumulative sum of probability
cumprob = np.cumsum(prob)

#Pixels for which cumulative is 0.9
i = cumprob.searchsorted(0.9)

#Sum of the areas of the pixels up to that one
area_90 = pixel_area[:i].sum()
area_90.to_value(u.deg**2)

UnitConversionError: '' (dimensionless) and 'sr' (solid angle) are not convertible